# Tập 1.4 · Bám quỹ đạo: robot chạy hình số 8 ♾️

[![Mở trên Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhpro82/tu-hoc-robot-tu-hanh/blob/main/tap14/tap14_bam_quy_dao.ipynb)

Series **Tự Học Robot Tự Hành** · Mùa 1 · ⏱ ~40 phút · Giáo trình: Principles of Robot Autonomy, [Chương 3](https://porabook.com/book/closed-loop-control/) (Ví dụ 3.1.2)

Robot đuổi theo một robot ảo đang vẽ số 8. Trong notebook này bạn:

1. Chạy lại đúng mô phỏng trong video.
2. **Tự viết** phẳng vi phân, điều khiển ảo và phép đổi về lệnh thật (3 bài), có ô chấm tự động.
3. Vẽ hình số 8 và sai số bằng chính code của bạn.

Chạy lần lượt từng ô từ trên xuống (Shift + Enter).

## 0. Chuẩn bị (chạy 1 lần)

In [ ]:
import os, sys
if "google.colab" in sys.modules:  # trên Colab: tải code về
    if not os.path.exists("/content/tu-hoc-robot-tu-hanh"):
        !git clone -q https://github.com/thanhpro82/tu-hoc-robot-tu-hanh.git /content/tu-hoc-robot-tu-hanh
    os.chdir("/content/tu-hoc-robot-tu-hanh/tap14")
sys.path[:0] = ["bai_tap", "sim", ".."]

import bam_quy_dao as goc          # mô phỏng trong video
from ve_do_thi import ve           # vẽ hình
from IPython.display import Image, display
from kiem_tra import cham          # bộ chấm
print("Sẵn sàng!")

## 1. Chạy lại số liệu trong video

In [ ]:
vs = [goc.tinh_san(*goc.tham_chieu(i * goc.dt)[2:])[0] for i in range(round(goc.T / goc.dt))]
print(f"Số 8 rộng {2 * goc.A} m, {goc.T:.0f} s, tốc độ robot ảo {min(vs):.2f}–{max(vs):.2f} m/s, kp = {goc.KP}, kd = {goc.KD}")
for ten, ss in [("Chỉ tính sẵn", goc.SS_TINH_SAN), ("Tính sẵn + sửa sai", goc.SS_BAM)]:
    n5 = round(5 / goc.dt)
    print(f"{ten:18s}: sai số đầu {ss[0] * 100:.0f} cm, lớn nhất {max(ss) * 100:.0f} cm, trung bình 5 s cuối {sum(ss[-n5:]) / n5 * 100:.1f} cm")

## 2. Bài tập

Phẳng vi phân: $v = \sqrt{\dot x^2+\dot y^2}$, $\theta = \operatorname{atan2}(\dot y, \dot x)$, $a = (\dot x\ddot x + \dot y\ddot y)/v$, $\omega = (\dot x\ddot y - \dot y\ddot x)/v^2$

Điều khiển ảo: $w_1 = \ddot x_d + k_p(x_d - x) + k_d(\dot x_d - \dot x)$ (tương tự $w_2$), rồi $[a, \omega]^T = J^{-1}[w_1, w_2]^T$ với $J = \begin{bmatrix}\cos\theta & -v\sin\theta\\ \sin\theta & v\cos\theta\end{bmatrix}$

| Bài | Hàm | Việc |
|---|---|---|
| **1** | `tinh_san(dx, dy, ddx, ddy)` | trả về `(v, θ, a, ω)` |
| **2** | `dieu_khien_ao(xd, yd, dxd, dyd, ddxd, ddyd, x, y, v, th, kp, kd)` | trả về `(w1, w2)` |
| **3** | `doi_lenh(w1, w2, th, v)` | trả về `(a, ω)` |

Thay từng dòng `raise NotImplementedError(...)` bằng code của bạn, chạy ô đó, rồi chạy ô chấm ngay bên dưới (✅ đạt · ❌ sai, kèm gợi ý · ⏳ chưa làm).

In [ ]:
import math


def tinh_san(dx, dy, ddx, ddy):
    """Bài 1 (phẳng vi phân): biết vận tốc (ẋ, ẏ) và gia tốc (ẍ, ÿ) của điểm cần đi qua, suy ra v, θ, a, ω.

        v = √(ẋ² + ẏ²)                θ = atan2(ẏ, ẋ)
        a = (ẋ·ẍ + ẏ·ÿ) / v           ω = (ẋ·ÿ − ẏ·ẍ) / v²
    Trả về (v, theta, a, omega)."""
    raise NotImplementedError("TODO Bài 1: trả về (v, theta, a, omega)")


def dieu_khien_ao(xd, yd, dxd, dyd, ddxd, ddyd, x, y, v, th, kp, kd):
    """Bài 2: điều khiển ảo (feedforward + feedback) cho hai đầu ra x, y.

    (xd, yd), (dxd, dyd), (ddxd, ddyd): vị trí, vận tốc, gia tốc của robot ảo trên hình số 8.
    (x, y, v, th): trạng thái robot thật; vận tốc thật là (v·cos θ, v·sin θ).
        w1 = ẍ_d + kp·(x_d − x) + kd·(ẋ_d − ẋ)
        w2 = ÿ_d + kp·(y_d − y) + kd·(ẏ_d − ẏ)
    Trả về (w1, w2)."""
    raise NotImplementedError("TODO Bài 2: trả về (w1, w2)")


def doi_lenh(w1, w2, th, v):
    """Bài 3: đổi (w1, w2) về lệnh thật (a, ω) bằng cách giải  J·[a, ω] = [w1, w2],

        J = [[cos θ, −v·sin θ],
             [sin θ,  v·cos θ]]
    Gợi ý: J⁻¹ có công thức gọn:  a = cos θ·w1 + sin θ·w2,   ω = (−sin θ·w1 + cos θ·w2) / v.
    Trả về (a, omega)."""
    raise NotImplementedError("TODO Bài 3: trả về (a, omega)")


In [ ]:
cham(globals())

## 3. Xem kết quả

In [ ]:
import loi_giai
try:
    ve(globals(), "ket_qua.png")
except NotImplementedError:
    print("Chưa xong cả 3 bài, vẽ bằng lời giải.")
    ve(loi_giai, "ket_qua.png")
display(Image("ket_qua.png"))

## 4. Thử thách

Đổi hệ số `kp`, `kd` (hoặc bỏ phần tính sẵn) rồi xem sai số thay đổi thế nào.

In [ ]:
import loi_giai
from kiem_tra import chay, _tb5
kp, kd = 1.0, 2.0                      # thử 16, 8 ...
bo_tinh_san = False                    # True: bỏ ẍ_d, ÿ_d

def dk(xd, yd, dxd, dyd, ddxd, ddyd, x, y, v, th, _kp, _kd):
    if bo_tinh_san:
        ddxd = ddyd = 0.0
    return loi_giai.dieu_khien_ao(xd, yd, dxd, dyd, ddxd, ddyd, x, y, v, th, kp, kd)

_, ss = chay(loi_giai.tinh_san, dk, loi_giai.doi_lenh)
print(f"kp = {kp}, kd = {kd}, bỏ tính sẵn = {bo_tinh_san}: sai số lớn nhất {max(ss) * 100:.1f} cm, 5 s cuối {_tb5(ss):.1f} cm")

## Câu hỏi tự kiểm tra

Tự trả lời trước, rồi mở đáp án ở mục *Câu hỏi tự kiểm tra* trong [README của tập](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh/tree/main/tap14).

Code cả series: [github.com/thanhpro82/tu-hoc-robot-tu-hanh](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh)